# textseg: build the page cache (CPU, run once)

Fetches the training and hard-negative pages from Danbooru, downscales them to
1536 on the long side, and writes `cache/pages/*.jpg` and `cache/labels.json`
(see `ml/textseg/build_cache.py`). The training notebook attaches this output,
so the fetch happens once.

**Settings:** Accelerator None, Internet on. **Input:** the code dataset.
Then Save Version -> Save & Run All.


In [ ]:
import glob, shutil, zipfile
from pathlib import Path

# The code dataset; unpack the zip if Kaggle kept it whole.
WORK = Path('/kaggle/working/comic-translator')
if not (WORK / 'ml/textseg/train.py').exists():
    found = glob.glob('/kaggle/input/**/ml/textseg/train.py', recursive=True)
    if found:
        shutil.copytree(Path(found[0]).parents[2], WORK, dirs_exist_ok=True)
    else:
        archive = glob.glob('/kaggle/input/**/comic-translator-ml.zip', recursive=True)
        assert archive, 'attach the comic-translator-ml dataset (Add Input)'
        zipfile.ZipFile(archive[0]).extractall(WORK)
assert (WORK / 'ml/textseg/train.py').exists()
print('code at', WORK)

In [ ]:
# Kaggle's CPU image has no onnxruntime; build_cache imports the eval module that needs it.
!pip install -q onnxruntime 2>&1 | tail -1

In [ ]:
!cd /kaggle/working/comic-translator && python ml/textseg/build_cache.py --out /kaggle/working/cache --workers 2

In [ ]:
# Make sure no raw originals are left in the output.
shutil.rmtree('/kaggle/working/cache/raw', ignore_errors=True)
shutil.rmtree('/kaggle/working/comic-translator', ignore_errors=True)
!du -sh /kaggle/working/cache